# 🧠 CalRetail — Automated Replenishment
## Goal
Compute optimal procurement levels and safety stock points using real per-supplier lead times,
real demand volatility, and a data-derived service level.

## Algorithmic Explanation
**Economic Order Quantity (EOQ) and Safety Stock calculation**
1. Annualise demand using the transaction history's real date span (not an assumed 30-day window).
2. Extract each product's real demand standard deviation and its real supplier's lead time
   (`suppliers.lead_time_days`) — previously every product used a flat 5-day lead time regardless
   of its actual supplier.
3. Apply EOQ (Wilson's Equation): `Q* = sqrt(2 * D * S / H)`.
4. Compute safety stock using a service-level z-score derived from the population's real
   historical stockout rate (`adaptive_thresholds.get_adaptive_service_level`), not a fixed 95%.



In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import json
import re
import math
from backend.utils.db import load_table  # SQLite-backed
warnings.filterwarnings('ignore')

# Set path to include parent directory
base_path = Path().resolve()
while not (base_path / 'data').exists() and base_path.parent != base_path:
    base_path = base_path.parent
processed_dir = base_path / 'data'
if str(base_path) not in sys.path:
    sys.path.insert(0, str(base_path))

print(f"Project root found at: {base_path}")
print(f"Data directory: {processed_dir}")

In [ ]:
from backend.utils.adaptive_thresholds import get_adaptive_service_level

tx = load_table('transactions')
prod = load_table('products')
suppliers = load_table('suppliers')

tx['transaction_date'] = pd.to_datetime(tx['transaction_date'])

# Annualise demand using the ACTUAL span of the transaction history — a fixed
# *365/30 multiplier applied to an all-time total would inflate a 3-year sum
# by ~12x, as if it were only a 30-day sum.
date_span_days = max((tx['transaction_date'].max() - tx['transaction_date'].min()).days, 1)
prod_annual = tx.groupby('product_id')['quantity'].sum().reset_index()
prod_annual['annual_demand'] = prod_annual['quantity'] * (365.0 / date_span_days)

# Standard deviation of daily demand (real demand volatility)
daily_vol = tx.groupby(['product_id', 'transaction_date'])['quantity'].sum().reset_index()
vol_stats = daily_vol.groupby('product_id')['quantity'].std().reset_index()
vol_stats.rename(columns={'quantity': 'daily_demand_std'}, inplace=True)

replenish_data = pd.merge(prod_annual, vol_stats, on='product_id', how='left')

# Real target service level (z-score), derived from the population's actual
# historical stockout rate — replaces a flat 95% / z=1.65 assumption.
SERVICE_LEVEL_Z = get_adaptive_service_level()
DEFAULT_LEAD_TIME = float(suppliers['lead_time_days'].median())

print(f"Core replenishment parameters engineered over a {date_span_days}-day history.")
print(f"Data-derived service level z-score: {SERVICE_LEVEL_Z}")

In [ ]:
def get_replenishment_parameters(product_id):
    row_data = replenish_data[replenish_data['product_id'] == product_id]
    if row_data.empty: return {"error": "Product reference missing"}
    
    r_item = row_data.iloc[0]
    D = max(1, r_item['annual_demand'])
    
    # Calculate setup and holding costs dynamically based on product price
    prod_row = prod[prod['product_id'] == product_id]
    price = prod_row.iloc[0]['price'] if not prod_row.empty else 100.0
    supplier_id = prod_row.iloc[0]['supplier_id'] if not prod_row.empty else None

    # Real supplier lead time for THIS product's actual supplier, not a flat
    # 5-day assumption applied to every SKU regardless of who supplies it.
    sup_row = suppliers[suppliers['supplier_id'] == supplier_id]
    lead_time_avg = float(sup_row.iloc[0]['lead_time_days']) if not sup_row.empty else DEFAULT_LEAD_TIME

    S = 150.0 # Fixed per-order administrative/setup cost (standard EOQ assumption)
    H = max(1.0, round(0.18 * price, 2))  # Annual holding cost: 18% of price
    
    # EOQ calculation
    eoq = np.sqrt((2 * D * S) / H)
    
    # Safety stock: service-level z-score derived from real historical stockout rates
    std_demand = r_item['daily_demand_std'] if not pd.isna(r_item['daily_demand_std']) else 1.0
    
    safety_stock = SERVICE_LEVEL_Z * std_demand * np.sqrt(lead_time_avg)
    rop = (D / 365) * lead_time_avg + safety_stock
    
    return {
        "product_id": product_id,
        "annual_demand": int(D),
        "safety_stock": int(safety_stock),
        "reorder_point": int(rop),
        "recommended_order_quantity": int(eoq),
        "lead_time_days": round(lead_time_avg, 1),
        "service_level_z": SERVICE_LEVEL_Z,
    }

sample_pid = replenish_data['product_id'].iloc[0]
repl_res = get_replenishment_parameters(sample_pid)
print("Replenishment recommendations output:\n", json.dumps(repl_res, indent=2))

In [ ]:
print("=== CALRETAIL AUTOMATED PROCUREMENT DESK ===")
print(f"Product identifier: {repl_res['product_id']}")
print(f"Annual Demand rate: {repl_res['annual_demand']} units")
print(f"Estimated Safety Stock Reserve: {repl_res['safety_stock']} units")
print(f"Reorder Trigger Point (ROP): {repl_res['reorder_point']} units")
print(f"Recommended Quantity to Order (EOQ): {repl_res['recommended_order_quantity']} units")
